# Session 2: From one neuron to a network

**259.075 Data-integrated Algorithmic Design Processes II** · Session 2 of 7

Last week you left a model standing at an error of about 50, and the diagnosis was that the
U-value of the wall sits in the data but not in the model. Today you hand it over. The error
does collapse, and the first thing we do afterwards is take the result apart, because it is
worth much less than it looks.

The second half is the more important one. A straight line, however many inputs you give it,
is still a straight line, and most things worth predicting are not straight. You will see what
goes wrong, watch the obvious repair fail, and then build the smallest thing that works.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

df = pd.read_csv("../data/gebaeude_wien.csv")
X = df[["kompaktheit", "u_wert"]].to_numpy()
y = df["hwb"].to_numpy()
print(X.shape, y.shape)
df[["kompaktheit", "u_wert", "hwb"]].describe().round(3)

## Two inputs

The model gains one weight:

    hwb = w1 · compactness + w2 · u_value + b

and the loop is the one you wrote last week, with a second gradient. Run it. Three thousand
steps, the same learning rate as last time.

In [ ]:
def mse(w, b, X, y):
    return float(np.mean((X @ w + b - y) ** 2))


def fit(X, y, lernrate=0.015, schritte=3000):
    w = np.zeros(X.shape[1])
    b = float(np.mean(y))
    verlauf = [mse(w, b, X, y)]
    for _ in range(schritte):
        fehler = X @ w + b - y
        w -= lernrate * 2 * (X.T @ fehler) / len(y)
        b -= lernrate * 2 * float(np.mean(fehler))
        verlauf.append(mse(w, b, X, y))
    return w, b, np.array(verlauf)


w_roh, b_roh, verlauf_roh = fit(X, y)
print(f"w = {w_roh.round(2)}, b = {b_roh:.1f}, error = {verlauf_roh[-1]:.1f}")

It has not arrived. After three thousand steps the error still sits above 40, and the run has
to grind through something like a hundred thousand steps before it gets anywhere near the
answer. Nothing is broken. The two inputs simply differ in how far they spread: compactness
varies with a standard deviation of about 1.6, the U-value with about 0.12, a factor of more
than thirteen. The gradient in the wide direction is correspondingly larger than in the narrow
one, and a single learning rate has to serve both.

<img src="img/slot-canyon.jpg" width="400">

*What the error landscape looks like from the inside. Photo: Jason J. Corneveaux,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Wirepass_slot_canyon.jpg),
CC BY-SA 3.0.*

> The negative gradient points downhill, not towards the minimum. In a narrow valley those are
> two very different directions.

The next cell draws the landscape over the two weights, with the path the run takes. Left the
raw data, right the same data with both inputs brought to the same spread. Same learning rate,
same number of steps, and the offset `b` held fixed in both, so that the drawing and the walk
happen on the same surface.

In [ ]:
def landschaft(X, y, b, w1_bereich, w2_bereich, n=70):
    w1 = np.linspace(*w1_bereich, n)
    w2 = np.linspace(*w2_bereich, n)
    Z = np.array([[mse(np.array([a, c]), b, X, y) for a in w1] for c in w2])
    return w1, w2, Z


def spur(X, y, b, lernrate, schritte):
    """Descent on the two weights only; b stays where it is, as in the picture."""
    w = np.zeros(X.shape[1])
    weg = [w.copy()]
    for _ in range(schritte):
        fehler = X @ w + b - y
        w -= lernrate * 2 * (X.T @ fehler) / len(y)
        weg.append(w.copy())
    return np.array(weg)


b_fest = float(np.mean(y))
X_skaliert = (X - X.mean(axis=0)) / X.std(axis=0)

fig = make_subplots(rows=1, cols=2, subplot_titles=("raw inputs", "same inputs, rescaled"))
for spalte, (daten, bereich1, bereich2) in enumerate(
        [(X, (-5, 25), (-40, 120)), (X_skaliert, (-5, 25), (-10, 50))], start=1):
    a, c, Z = landschaft(daten, y, b_fest, bereich1, bereich2)
    weg = spur(daten, y, b_fest, 0.015, 200)
    fig.add_trace(go.Contour(x=a, y=c, z=np.log10(Z), colorscale="Viridis", showscale=False,
                             contours=dict(showlines=False)), row=1, col=spalte)
    fig.add_trace(go.Scatter(x=weg[:, 0], y=weg[:, 1], mode="lines+markers",
                             line=dict(color="crimson", width=2), marker=dict(size=3)),
                  row=1, col=spalte)
fig.update_layout(height=470, showlegend=False,
                  title="200 steps at the same learning rate, log error, b held fixed")
fig.update_xaxes(title_text="w1 (compactness)")
fig.update_yaxes(title_text="w2 (u-value)")
fig.show()

On the left the path crawls along a ridge; on the right it walks in. The repair is to measure
both inputs in the same currency: subtract the mean, divide by the standard deviation. Every
input then has mean zero and spread one, and the landscape is far better conditioned. Not
perfectly round, because the two inputs are still slightly correlated, but close enough that
one learning rate suits both directions.

<img src="img/metre-etalon.jpg" width="330">

*One of the standard metres set into the walls of Paris in 1791, rue de Vaugirard, so that
everybody measured with the same stick. Photo: Airair,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:M%C3%A8tre_%C3%A9talon_(36,_rue_de_Vaugirard,_Paris).jpg),
CC BY-SA 3.0.*

> Rescaling is not a condition for fitting a model. It is a condition for fitting it before
> lunch, and for being able to compare the weights afterwards.

The price is that the weights now mean something else: each one says how much the target moves
when its input goes up by one standard deviation. That is a perfectly good statement, but it is
not a statement anybody can check against a building code. Your job is to convert them back.

In [ ]:
def normiere(X):
    mittel, streuung = X.mean(axis=0), X.std(axis=0)
    return (X - mittel) / streuung, mittel, streuung


def zurueck(w_n, b_n, mittel, streuung):
    """Weights from the normalised run, expressed in the original units."""

    w = w_n / streuung
    b = b_n - float(np.sum(w_n * mittel / streuung))

    return w, float(b)


Xn, mittel, streuung = normiere(X)
w_n, b_n, verlauf_n = fit(Xn, y, lernrate=0.015, schritte=400)
w_o, b_o = zurueck(w_n, b_n, mittel, streuung)

print(f"400 steps, error {verlauf_n[-1]:.5f}")
print(f"per unit of compactness: {w_o[0]:.4f}")
print(f"per W/m2K of u-value   : {w_o[1]:.4f}")
print(f"offset                 : {b_o:.4f}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=verlauf_roh, mode="lines", name="raw, 3000 steps"))
fig.add_trace(go.Scatter(y=verlauf_n, mode="lines", name="rescaled, 400 steps"))
fig.update_layout(height=400, xaxis_title="step", yaxis_title="error", yaxis_type="log",
                  xaxis_type="log", title="what rescaling buys you")
fig.show()

Nine, sixty-two, eighteen, to four decimal places. Those are the numbers in the formula that
produced the column in the first place, printed openly in `data/fetch_ogd.py`:

    hwb = 18 + 62 · u_value + 9 · compactness

The last digits are off by a few thousandths, and that is not the model's doing: the CSV stores
compactness rounded to three decimals and the target to two, so the two sides no longer agree
exactly.

The model therefore did not discover anything. It fished back out a calculation rule that
somebody had put in. That rule carries no measurement noise, which is why the recovery is this
clean; it says nothing about the accuracy of the footprints underneath, and the U-values are
textbook values per construction period, not measurements. Against measured consumption none of
this would look remotely like it does here.

> A model that reproduces its own generating formula has learned arithmetic, not buildings.

What is worth keeping from these twenty minutes is the rescaling. You need it for the rest of
the term, from the moment geometry, material and use sit in the same model.

The next cell shows why the fit is so exact: the data lies on a plane, and the model is that
plane. Turn it.

In [ ]:
k = np.linspace(X[:, 0].min(), X[:, 0].max(), 30)
u = np.linspace(X[:, 1].min(), X[:, 1].max(), 30)
K, U = np.meshgrid(k, u)
Z = w_o[0] * K + w_o[1] * U + b_o

fig = go.Figure()
fig.add_trace(go.Surface(x=k, y=u, z=Z, colorscale="Viridis", opacity=0.55, showscale=False))
fig.add_trace(go.Scatter3d(x=X[:, 0], y=X[:, 1], z=y, mode="markers",
                           marker=dict(size=2, color=df["startjahr"], colorscale="Turbo"),
                           name="buildings"))
fig.update_layout(height=620, title="the model is a plane, and the buildings sit on it",
                  scene=dict(xaxis_title="compactness", yaxis_title="u-value",
                             zaxis_title="heating demand"))
fig.show()

## Where a plane is not enough

Everything so far was straight because the target was made straight. The next dataset is not.
The compactness values are the real ones from the Vienna footprints, the target is invented,
and here is the rule that invents it:

    target = 40 + 9 · compactness + 45 · sin(0.8 · compactness)

No physics is claimed. It is a curve, and that is the whole point. Fit the best straight line
to it and look at what is left over.

In [ ]:
kx = np.sort(df["kompaktheit"].to_numpy())
ky = 40 + 9 * kx + 45 * np.sin(0.8 * kx)

steigung, achse = np.polyfit(kx, ky, 1)
gerade = steigung * kx + achse
rest = ky - gerade

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
                    subplot_titles=("the best straight line there is", "what it leaves over"))
fig.add_trace(go.Scatter(x=kx, y=ky, mode="markers", name="data",
                         marker=dict(size=4, color="#4c78a8")), row=1, col=1)
fig.add_trace(go.Scatter(x=kx, y=gerade, mode="lines", name="straight model",
                         line=dict(width=4, color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=kx, y=rest, mode="markers", name="residual",
                         marker=dict(size=4, color="#8b5cf6")), row=2, col=1)
fig.add_hline(y=0, row=2, col=1, line_width=1)
fig.update_layout(height=620, xaxis2_title="compactness")
fig.show()

The residuals are not scattered around zero, they swing. Whole stretches of the data sit above
the line, whole stretches below. No learning rate and no amount of patience fixes that, because
the model has no shape to give.

> When residuals show a pattern, the model is missing a form, not a better fit.

## Two straight lines are one straight line

The obvious repair is to stack: feed the output of one linear layer into another.

    h = a · x + p
    y = c · h + q

Pull the slider through the twenty-five combinations of `a` and `c`. Whatever you pick, the
result stays a straight line, and the two lines of arithmetic underneath say why:

    y = c · (a · x + p) + q = (c · a) · x + (c · p + q)

A product of two numbers is a number. Stacking linear layers buys you nothing at all, and that
is the central reason networks contain something else in between.

In [ ]:
xs = np.linspace(0, 10, 100)
paare = [(a, c) for a in [-1.5, -0.5, 0.5, 1.5, 2.5] for c in [-1.5, -0.5, 0.5, 1.5, 2.5]]
alle = np.array([c * (a * xs + 1.0) + 2.0 for a, c in paare])

frames = [go.Frame(name=f"{a},{c}",
                   data=[go.Scatter(x=xs, y=c * (a * xs + 1.0) + 2.0, mode="lines",
                                    line=dict(width=4, color="crimson"))],
                   traces=[0],
                   layout=go.Layout(title=f"a = {a}, c = {c} — still a straight line"))
          for a, c in paare]

fig = go.Figure(data=[go.Scatter(x=xs, y=alle[0], mode="lines",
                                 line=dict(width=4, color="crimson"))],
                frames=frames)
fig.update_layout(height=430, xaxis_title="x", yaxis_title="y",
                  yaxis_range=[alle.min() - 5, alle.max() + 5],
                  title=f"a = {paare[0][0]}, c = {paare[0][1]} — still a straight line",
                  sliders=[dict(active=0, currentvalue=dict(prefix="a, c = "),
                                steps=[dict(method="animate", label=f"{a},{c}",
                                            args=[[f"{a},{c}"],
                                                  dict(mode="immediate",
                                                       frame=dict(duration=0, redraw=True))])
                                       for a, c in paare])])
fig.show()

## The kink

What goes in between is almost insultingly simple. Take the output of a neuron and set
everything below zero to zero:

    relu(z) = max(0, z)

That is the whole function. It is called a *rectified linear unit*, ReLU for short, and it
gives you one bend at one place. One neuron with a ReLU is a ramp that starts somewhere and
climbs at some rate.

Several of them side by side, each with its own starting point, added up with their own
weights: that is a curve. The next cell places the bends by repeated halving, so that going
from four neurons to five keeps the four bends you already had and adds one in between. Because
the second layer is still linear, the output weights can then be solved for exactly instead of
trained. Pull the slider from one neuron to sixteen.

The first bend sits at the smallest compactness in the data, so there is no stretch where every
ramp is still flat; with one neuron the model is exactly a straight line, which is the right
place to start from.

> Neurons in a layer sit next to each other, not behind each other. The curve is their sum.

In [ ]:
def knickfolge(x, anzahl):
    """Bends by repeated halving: the first n are always a subset of the first n+1."""
    lo, hi = float(x.min()), float(x.max())
    folge, ebene = [lo], 1
    while len(folge) < anzahl:
        for i in range(1, 2 ** ebene, 2):
            folge.append(lo + (hi - lo) * i / 2 ** ebene)
            if len(folge) == anzahl:
                break
        ebene += 1
    return np.array(folge)


def rampen(x, knicke):
    """One column per neuron: the ramp max(0, x - bend), plus a constant column."""
    H = np.maximum(0.0, x[:, None] - knicke[None, :])
    return np.hstack([H, np.ones((len(x), 1))])


def netz(x, y, anzahl):
    knicke = knickfolge(x, anzahl)
    gewichte, *_ = np.linalg.lstsq(rampen(x, knicke), y, rcond=None)
    return knicke, gewichte


zahlen = list(range(1, 17))
laeufe = {n: netz(kx, ky, n) for n in zahlen}


def kurve(n, stellen):
    knicke, gewichte = laeufe[n]
    return rampen(stellen, knicke) @ gewichte


def beitraege(n, stellen):
    """What each neuron adds, on its own, without the constant."""
    knicke, gewichte = laeufe[n]
    H = np.maximum(0.0, stellen[:, None] - knicke[None, :])
    return [H[:, i] * gewichte[i] for i in range(n)]


def stuetzen(n):
    """A piecewise linear function is exact on its knots; no need for 1073 x-values."""
    return np.unique(np.concatenate([laeufe[n][0], [kx.min(), kx.max()]]))


fehler = {n: float(np.mean((kurve(n, kx) - ky) ** 2)) for n in zahlen}
alle_beitraege = np.concatenate([np.concatenate(beitraege(n, stuetzen(n))) for n in zahlen])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                    subplot_titles=("the sum of the ramps, plus a constant",
                                    "what each neuron contributes"))
fig.add_trace(go.Scatter(x=kx, y=ky, mode="markers", name="data",
                         marker=dict(size=4, color="#4c78a8")), row=1, col=1)
fig.add_trace(go.Scatter(x=stuetzen(1), y=kurve(1, stuetzen(1)), mode="lines", name="network",
                         line=dict(width=4, color="crimson")), row=1, col=1)
for spur_i, linie in enumerate(beitraege(1, stuetzen(1)) + [None] * 15):
    s = stuetzen(1)
    fig.add_trace(go.Scatter(x=s if linie is not None else s,
                             y=linie if linie is not None else np.full_like(s, np.nan),
                             mode="lines", showlegend=False,
                             line=dict(width=1.5, color="#d8a24a")), row=2, col=1)

frames = []
for n in zahlen:
    s = stuetzen(n)
    linien = beitraege(n, s)
    daten = [go.Scatter(x=s, y=kurve(n, s), mode="lines",
                        line=dict(width=4, color="crimson"))]
    daten += [go.Scatter(x=s, y=l, mode="lines", line=dict(width=1.5, color="#d8a24a"))
              for l in linien]
    daten += [go.Scatter(x=s, y=np.full_like(s, np.nan), mode="lines",
                         line=dict(width=1.5, color="#d8a24a"))
              for _ in range(16 - n)]
    frames.append(go.Frame(name=str(n), data=daten, traces=[1] + list(range(2, 18)),
                           layout=go.Layout(
                               title=f"{n} neuron{'s' if n > 1 else ''}, "
                                     f"error {fehler[n]:.1f}")))

fig.frames = frames
fig.update_layout(height=700, showlegend=False,
                  title=f"1 neuron, error {fehler[1]:.1f}",
                  sliders=[dict(active=0, currentvalue=dict(prefix="neurons: "),
                                steps=[dict(method="animate", label=str(n),
                                            args=[[str(n)],
                                                  dict(mode="immediate",
                                                       frame=dict(duration=0, redraw=False))])
                                       for n in zahlen])])
fig.update_xaxes(range=[kx.min() - 0.3, kx.max() + 0.3])
fig.update_yaxes(range=[ky.min() - 15, ky.max() + 15], row=1, col=1)
fig.update_yaxes(range=[alle_beitraege.min() - 20, alle_beitraege.max() + 20], row=2, col=1)
fig.update_xaxes(title_text="compactness", row=2, col=1)
fig.show()

print("neurons and error:", {n: round(fehler[n], 1) for n in zahlen})

The error falls every time you add a neuron, and it has to: the bends you had are still there,
so the new model can do everything the old one could and one thing more. Had the bends been
redistributed at every step instead, the error would sometimes get worse, which is worth
knowing before you read a paper that shows a curve like this.

The lower panel shows contributions, not activations. Each line is one ramp already multiplied
by its output weight, which is why some of them point downwards. The constant is not drawn; the
sum of the lines plus that constant is the red curve above.

Sixteen bends follow the curve closely. Hold that thought until next week, when we ask what
happens if you keep going until the model passes through every single point.

## Writing it once instead of sixteen times

With sixteen neurons you no longer want sixteen lines of arithmetic. Put each neuron's weights
in a row of a matrix and the whole forward pass is one multiplication:

    h = relu(x · W1ᵀ + b1)
    y = h · W2 + b2

`W1` has one row per neuron, `b1` one bias per neuron, and `W2` the weight each neuron
contributes to the output. Nothing new happens here; the same additions are written more
compactly. The cell below runs the same sixteen-neuron network once in a loop and once as a
matrix product, and checks that the two agree.

<img src="img/brunsviga.jpg" width="380">

*Before this was a matrix product it was a morning's work. Brunsviga 15, around 1930. Photo:
CEphoto, Uwe Aranas,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Mechanical-calculator-Brunsviga-15-01.jpg),
CC BY-SA 3.0.*

In [ ]:
knicke, gewichte = laeufe[16]

W1 = np.ones((16, 1))                 # one row per neuron, one input
b1 = -knicke                          # the bend of each neuron
W2 = gewichte[:16].reshape(16, 1)     # what each neuron contributes
b2 = gewichte[16]

einzel = np.array([sum(max(0.0, w * xi + b) * v for w, b, v
                       in zip(W1[:, 0], b1, W2[:, 0])) + b2 for xi in kx])
matrix = np.maximum(0.0, kx[:, None] @ W1.T + b1) @ W2 + b2

print("loop and matrix agree:", np.allclose(einzel, matrix.ravel()))
print("W1", W1.shape, " b1", b1.shape, " W2", W2.shape)

## The same thing in PyTorch

From next week you write this yourself, so look at it once now. Ten lines, and every one of
them has a counterpart in what you just did by hand: the model is the two layers with the ReLU
between them, the loss is the mean squared error from session 1, the optimiser is gradient
descent with a better step rule, and the loop is the loop.

Keep these names. They come back in session 3 with three of the lines missing.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
Xt = torch.tensor(kx, dtype=torch.float32).unsqueeze(1)
Yt = torch.tensor(ky, dtype=torch.float32).unsqueeze(1)
x_m, x_s = Xt.mean(), Xt.std()
y_m, y_s = Yt.mean(), Yt.std()          # the target gets rescaled too, and undone below

modell = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
verlust = nn.MSELoss()
optimierer = torch.optim.Adam(modell.parameters(), lr=0.05)

for schritt in range(2000):
    vorhersage = modell((Xt - x_m) / x_s)
    fehler_t = verlust(vorhersage, (Yt - y_m) / y_s)
    optimierer.zero_grad()
    fehler_t.backward()
    optimierer.step()

mit_torch = (modell((Xt - x_m) / x_s) * y_s + y_m).detach().numpy().ravel()
print(f"by hand, bends fixed by halving, output weights solved : error {fehler[16]:.2f}")
print(f"pytorch, both layers trained for 2000 steps           : error "
      f"{np.mean((mit_torch - ky) ** 2):.2f}")

In [ ]:
s16 = stuetzen(16)
fig = go.Figure()
fig.add_trace(go.Scatter(x=kx, y=ky, mode="markers", name="data",
                         marker=dict(size=4, color="#4c78a8")))
fig.add_trace(go.Scatter(x=s16, y=kurve(16, s16), mode="lines", name="by hand, solved exactly",
                         line=dict(width=4, color="crimson")))
fig.add_trace(go.Scatter(x=kx, y=mit_torch, mode="lines", name="pytorch, trained",
                         line=dict(width=3, color="#d8a24a", dash="dash")))
fig.update_layout(height=470, xaxis_title="compactness", yaxis_title="target",
                  title="same size of network, two ways of getting the weights")
fig.show()

The hand-built version wins, and by a wide margin. That is not an argument against training. It
is an argument about this particular problem: one input, no noise, and a second layer that is
still linear, so its weights can be solved for in closed form rather than searched for. Sixteen
bends spread by halving cover a one-dimensional range perfectly well, and there is nothing left
for a search to find. PyTorch has to move both layers at once and stops wherever two thousand
steps leave it.

Put two inputs in and the argument reverses. Bends by halving would need sixteen times sixteen
of them to cover the plane as finely, and in eight dimensions the idea collapses entirely.
Learning where the bends go is what makes networks work on anything wider than a line.

Do not read that as *everything is learned*, either. The width of the layer, the activation
function and the number of layers are choices you made before training started, and they decide
what the network is able to represent at all. Training picks the weights inside that choice.

## Tests

This cell has to run green before you submit. It checks your back-conversion twice: once
against the formula the data came from, and once against a set of weights that has nothing to
do with buildings, so that a function which simply returns the right answer for this one
dataset does not pass.

In [ ]:
Xn_, mittel_, streuung_ = normiere(X)
w_n_, b_n_, _ = fit(Xn_, y, lernrate=0.015, schritte=400)

erwartet = Xn_ @ w_n_ + b_n_                      # computed before anything is handed over
w_t, b_t = zurueck(w_n_.copy(), float(b_n_), mittel_.copy(), streuung_.copy())
w_t, b_t = np.asarray(w_t, dtype=float), float(b_t)

assert np.isfinite(w_t).all() and np.isfinite(b_t), "parameters ran away"
assert np.allclose(X @ w_t + b_t, erwartet, atol=1e-6), (
    "the converted weights do not reproduce the normalised model")
assert np.allclose(w_t, [9.0, 62.0], rtol=0.01), f"weights off: {w_t.round(3)}"
assert abs(b_t - 18.0) < 0.18, f"offset off: {b_t:.3f}"

# same question, arbitrary numbers, nothing to do with this dataset
probe_m, probe_s = np.array([100.0, -3.0]), np.array([20.0, 0.5])
probe_w, probe_b = np.array([2.0, -7.0]), 5.0
p_w, p_b = zurueck(probe_w.copy(), probe_b, probe_m.copy(), probe_s.copy())
assert np.allclose(p_w, probe_w / probe_s), f"second case, weights: {np.asarray(p_w).round(4)}"
assert abs(float(p_b) - (probe_b - float(np.sum(probe_w * probe_m / probe_s)))) < 1e-9, (
    f"second case, offset: {float(p_b):.4f}")

print(f"all green: {w_t[0]:.4f} per unit of compactness, {w_t[1]:.4f} per W/m2K, "
      f"offset {b_t:.4f}")

## Where you stand now

You can put several inputs into one model, you know why bringing them to a common spread makes
the difference between four hundred steps and a hundred thousand, and you can read the weights
back in units somebody can argue with. You have seen a straight model fail on curved data and
recognised the failure in the residuals rather than in the error number. And you can build a
curve out of kinks, which is the entire architectural idea behind every network in this course.

What you should not take away is the collapse of the error in the first half. That was
arithmetic recovering arithmetic.

Next session the target carries noise, you write PyTorch yourself, and the model that fits the
training data perfectly turns out to be the worse one.

---

### Terms to learn

The flashcard app has these, together with the terms from the other sessions, at
[gabriel-wurzer.github.io/data2/webapps/lernkarten](https://gabriel-wurzer.github.io/data2/webapps/lernkarten/).

| term | in one sentence |
|---|---|
| normalisation | shifting each input to mean zero and spread one, so one learning rate suits all directions |
| standard deviation | the usual distance of a value from the mean of its column |
| feature | one input column; choosing them is a design decision, not a technicality |
| layer | several neurons computing side by side from the same inputs |
| ReLU | max(0, z); one kink, and the reason a network is more than a line |
| activation function | the non-linear step between two layers |
| weight matrix | one row per neuron, so a whole layer is a single multiplication |
| forward pass | computing the output of the network from its inputs |
| optimiser | the rule that turns gradients into steps; Adam is one |
| residual pattern | structure left in the residuals, meaning the model lacks form |

## Homework

Ten points: four for the run, six for what you make of it. Submit this notebook, run from top
to bottom, with your text in the last cell. If you used lines from the solution notebook, say
so in one line at the top of your text. That costs nothing, see the course rules; a submission
that is missing costs everything.

Above this heading, the only thing to fill in is the `YOUR TURN` gap from class, by yourself
or from the solution. Below it, change only the lines under `# YOUR CHOICE`, and write your
answers into the last cell. Before you submit, choose Kernel, Restart Kernel and Run All Cells,
and check that everything still runs.

The four points for the run: the notebook runs from top to bottom without an error (one), what
the task asks you to show is there (one), and every number you give about your own run either
appears in your output, rounded as printed, or is worked out or counted from it in a step you
write down (two). Numbers the questions give you, numbers in an example from outside the
notebook, and a forecast you mark as your own expectation with a reason, are exempt. The six points for the interpretation are one per question below, no halves. A
sentence that would fit any model earns nothing; a sentence that names what you saw and says
what it means earns the point, also with borrowed code.

### The task

Take the PyTorch network from above and fit it to the curved data. The first cell trains it with
five different numbers of neurons, at two learning rates that you choose. Everything else stays
as it is, including the two thousand steps and the starting seed.

1. Pick two learning rates between 0.001 and 0.2, the larger at least three times the smaller,
   not both from the examples in class, and enter them. Run the cell once. That gives you a table of ten errors; you do not have to
   search further.
2. The second cell draws your best network on a range wider than the data and prints what it
   says at two places outside. Keep the figure.

The table will not be tidy. That is part of the result, not a reason to leave numbers out.

In [ ]:
def versuch(neuronen, lernrate, schritte=2000, samen=0):
    torch.manual_seed(samen)
    m = nn.Sequential(nn.Linear(1, neuronen), nn.ReLU(), nn.Linear(neuronen, 1))
    opt = torch.optim.Adam(m.parameters(), lr=lernrate)
    for _ in range(schritte):
        f = verlust(m((Xt - x_m) / x_s), (Yt - y_m) / y_s)
        opt.zero_grad()
        f.backward()
        opt.step()
    with torch.no_grad():
        aus = (m((Xt - x_m) / x_s) * y_s + y_m).numpy().ravel()
    return m, float(np.mean((aus - ky) ** 2))


neuronenzahlen = [2, 4, 8, 16, 64]

# YOUR CHOICE: two learning rates between 0.001 and 0.2, the larger at least three times the
# smaller
lernraten = [None, None]

assert None not in lernraten, "enter two learning rates"
assert all(0.001 <= lr <= 0.2 for lr in lernraten), "both rates between 0.001 and 0.2"
assert max(lernraten) >= 3 * min(lernraten), "the larger rate at least three times the smaller"
ergebnisse = {}
for lr in lernraten:
    for n in neuronenzahlen:
        ergebnisse[(n, lr)] = versuch(n, lr)
        print(f"learning rate {lr:<7} {n:3d} neurons  error {ergebnisse[(n, lr)][1]:9.3f}")
    print()
print(f"by hand, 16 bends solved exactly: error {fehler[16]:.3f}")
print(f"the error is a mean of squared misses, so its unit is the target's unit squared; "
      f"on the scaled target it would have no unit and be {float(y_s) ** 2:.0f} times smaller")

In [ ]:
bestes = min(ergebnisse, key=lambda s: ergebnisse[s][1])
m_best = ergebnisse[bestes][0]
regel = lambda k: 40 + 9 * k + 45 * np.sin(0.8 * k)


def netz_bei(stellen):
    with torch.no_grad():
        t = torch.tensor(stellen, dtype=torch.float32).unsqueeze(1)
        return (m_best((t - x_m) / x_s) * y_s + y_m).numpy().ravel()


weit = np.linspace(1.0, 16.0, 400)
fig = go.Figure()
fig.add_vrect(x0=kx.min(), x1=kx.max(), fillcolor="#4c78a8", opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=weit, y=regel(weit), mode="lines", name="the rule",
                         line=dict(width=2, color="#8b95a8", dash="dot")))
fig.add_trace(go.Scatter(x=kx, y=ky, mode="markers", name="data",
                         marker=dict(size=4, color="#4c78a8")))
fig.add_trace(go.Scatter(x=weit, y=netz_bei(weit), mode="lines", name="your network",
                         line=dict(width=3, color="crimson")))
fig.update_layout(height=470, xaxis_title="compactness", yaxis_title="target",
                  title=f"best run: {bestes[0]} neurons, learning rate {bestes[1]}, "
                        f"error {ergebnisse[bestes][1]:.3f}; shaded where there is data")
fig.show()

print(f"data from compactness {kx.min():.2f} to {kx.max():.2f}")
for k in (2.0, 15.0):
    print(f"at compactness {k:4.1f}: network {netz_bei(np.array([k]))[0]:7.1f}, "
          f"rule {regel(k):7.1f}")

### Your interpretation

1. **Size.** Give your lowest error and its square root, the typical miss. Compare it with the sixteen bends placed by hand.
2. **More neurons.** At each of your two learning rates, did more neurons always give a lower error? Answer from your table, yes or no for each rate, with the numbers. Where the answer is no, name the step at which it breaks.
3. **Units.** The error is a mean of squared misses, computed after the scaling of the target was undone. In what units is it? The first cell prints by what factor it would shrink on the scaled target. The bends by hand were measured on the unscaled target; say why a number from the scaled target could not be compared with theirs.
4. **Range.** Give network and rule at the two printed points outside the data, and say how far your network departs from the rule on each side. The figure shows you what happens in between.
5. **One run that went wrong.** Your table has two runs for each number of neurons, one per learning rate. Find the pair whose errors differ most and give a possible reason. The table cannot prove it, and saying so is part of a good answer. If no pair differs much, say that, with the numbers, and what it suggests.
6. **Design consequence.** The first half of this session reached an error of nearly zero on the Vienna buildings. What does that number tell you about how well a heating-demand model would do on a real building? Answer it as if a client had asked.

*Your text here.*